# Fuel demand from vessel telemetry

This notebook models fuel demand from a vessel's telemetry stream, two ways. The first is
inference: an OLS fit with HC3 standard errors and an elastic net fit on the same design, to see
which signals carry the effect. The second is prediction: six sklearn pipelines compared by
cross-validation, the best one tuned with random search, then scored on the final 20 percent of
the voyage. Residuals are mapped along the track so a systematic error over one leg is visible.

The telemetry is synthetic: 6,000 one-minute records along a transatlantic route with speed,
heading, draft, wind and wave state, sea temperature, engine load and a fuel demand column that
is a nonlinear function of those with noise. Column names are read from the table, so a real
export with a different set of sensors needs only a different `target`.

In [1]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
warnings.filterwarnings("ignore")

DATA = ROOT / "data" / "maritime"
EXPORTS = ROOT / "exports" / "maritime"
DATA.mkdir(parents=True, exist_ok=True)
EXPORTS.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
print("repo root:", ROOT)

repo root: /mnt/c/Users/joogl/OneDrive/Documents/VisualStudioCodeProjects/Fortress-GIS-Projects


In [2]:
from datasets.synthetic import synthetic_vessel_telemetry
from fortress_gis.domains import maritime as mar
from fortress_gis.viz.kepler import KeplerMapBuilder, kepler_available

source = DATA / "vessel_telemetry.parquet"
if not source.exists():
    synthetic_vessel_telemetry(6000).to_parquet(source)
telemetry = pd.read_parquet(source)
print(telemetry.shape)
telemetry.head()

(6000, 18)


,timestamp,longitude,latitude,fuel_demand_kg_h,speed_over_ground_kn,speed_through_water_kn,heading_deg,rudder_angle_deg,draft_fore_m,draft_aft_m,wind_speed_kn,wind_dir_deg,true_wind_speed_kn,wave_height_m,wave_period_s,wave_dir_deg,water_depth_m,sea_temp_c
0,1706745600,-9.500000,38.600000,327.500011,15.020460,15.184493,173.286719,-1.580283,9.496968,10.182226,9.907286,353.286719,5.401149,1.555905,7.036096,350.914107,3018.760650,18.060007
1,1706745660,-9.509809,38.601155,214.946448,12.732641,12.347427,173.286732,3.930177,9.495932,10.224730,11.265831,353.393564,7.446045,1.320007,6.970614,347.739948,3019.042124,17.216390
2,1706745720,-9.519618,38.602309,294.861178,14.229996,14.688262,173.286774,2.677632,9.476839,10.192701,11.538456,353.500437,7.269486,1.687480,6.440171,350.957181,3003.150980,18.298250
3,1706745780,-9.529428,38.603464,286.963889,13.747535,13.585731,173.286842,-2.892254,9.476639,10.194284,14.431917,353.607337,10.307721,2.187529,7.194745,345.071474,2990.667432,17.842170
4,1706745840,-9.539237,38.604618,251.813374,13.815567,13.506000,173.286939,1.386626,9.486584,10.205235,10.291766,353.714263,6.147211,1.308549,6.348896,351.155406,2939.163085,18.280564


## Build the design

`prepare_design` converts the timestamp, separates numeric from categorical columns (a column
with 20 or fewer distinct values is treated as categorical), one-hot encodes the categoricals
with the first level dropped, and returns the encoded frame with the feature list. Coordinates
and the timestamp are excluded from the predictors by default.

In [3]:
TARGET = "fuel_demand_kg_h"
design = mar.prepare_design(telemetry, target=TARGET)
print("numeric:", design.numeric)
print("categorical:", design.categorical)
print("features after encoding:", len(design.features))
design.frame[design.features].describe().T.round(2).head(12)

numeric: ['speed_over_ground_kn', 'speed_through_water_kn', 'heading_deg', 'rudder_angle_deg', 'draft_fore_m', 'draft_aft_m', 'wind_speed_kn', 'wind_dir_deg', 'true_wind_speed_kn', 'wave_height_m', 'wave_period_s', 'wave_dir_deg', 'water_depth_m', 'sea_temp_c', 'is_night']
categorical: []
features after encoding: 15


,count,mean,std,min,25%,50%,75%,max
speed_over_ground_kn,6000.0,14.00,1.51,10.52,12.67,13.99,15.34,17.55
speed_through_water_kn,6000.0,13.83,1.61,9.78,12.51,13.85,15.16,18.27
heading_deg,6000.0,178.31,3.17,173.29,175.20,178.31,181.47,182.79
rudder_angle_deg,6000.0,-0.00,3.10,-13.44,-1.90,0.01,1.90,11.97
draft_fore_m,6000.0,9.20,0.17,8.86,9.05,9.20,9.35,9.55
draft_aft_m,6000.0,9.90,0.17,9.55,9.75,9.90,10.05,10.25
wind_speed_kn,6000.0,8.66,4.69,0.00,4.60,9.20,12.62,20.33
wind_dir_deg,6000.0,159.44,140.81,0.01,38.81,59.56,311.06,359.98
true_wind_speed_kn,6000.0,5.49,4.73,-4.69,1.50,6.22,9.34,16.15
wave_height_m,6000.0,1.44,0.64,0.00,0.93,1.47,1.93,3.19


## Run the pipeline

`run_maritime_pipeline` fits everything in one call and returns a `MaritimeResult`: the design,
the OLS and elastic net fits, the model leaderboard, the tuned best model with its search
results, holdout metrics, permutation importances, and holdout residuals joined to the
coordinates. The sections below unpack it. Tuning runs `n_iter * cv` fits in parallel across
cores; with 12 candidates and 5 folds this cell takes about a minute on 12 cores.

In [4]:
result = mar.run_maritime_pipeline(telemetry, target=TARGET, cv=5, n_iter=12)
result.summary().round(3)

rows                  6000.000
features                15.000
ols_r2_adj               0.922
penalised_nonzero       15.000
best_model_rmse_cv      12.796
holdout_rmse            15.802
holdout_r2               0.754
dtype: float64

## Inference: OLS and elastic net

The OLS coefficients answer "how much does fuel demand move per unit of each predictor, holding
the rest fixed". The elastic net (alpha 0.05, L1 weight 0.05) is fit on z-scored features so the
penalty treats them equally, and it shrinks coefficients toward zero. To compare the two, the
OLS estimates are multiplied by each feature's standard deviation, which puts both on the
"kg/h per one standard deviation" scale. Collinear pairs (speed over ground and through water,
wind speed and true wind speed) trade weight between them under the penalty, so read those as
groups rather than one at a time. The penalised fit has no standard errors, so its table has
coefficients only.

In [5]:
ols, penalised = result.ols, result.penalised
print(ols.metrics())
coef = ols.coefficients().set_index("term")
coef.sort_values("p_value").round(4).head(12)

kind                   ols
n_obs                 6000
n_features              15
r2                0.921911
r2_adj            0.921715
aic           48210.970666
bic           48318.162902
dtype: object


,estimate,std_error,statistic,p_value,ci_low,ci_high
term,,,,,,
is_night,15.6439,0.3986,39.2441,0.0000,14.8626,16.4252
speed_through_water_kn,26.3915,0.7159,36.8627,0.0000,24.9883,27.7947
wave_height_m,15.8105,0.6346,24.9141,0.0000,14.5667,17.0543
const,-342.8846,30.2785,-11.3243,0.0000,-402.2294,-283.5397
sea_temp_c,-1.6229,0.1928,-8.4192,0.0000,-2.0007,-1.2451
true_wind_speed_kn,1.7093,0.2783,6.1430,0.0000,1.1639,2.2547
wind_speed_kn,-1.7787,0.3050,-5.8309,0.0000,-2.3766,-1.1808
wind_dir_deg,-0.0061,0.0022,-2.7156,0.0066,-0.0105,-0.0017
draft_aft_m,15.9314,6.3135,2.5234,0.0116,3.5572,28.3057


In [6]:
pen = penalised.coefficients().set_index("term")
scale = pd.Series(penalised.extra["feature_scale"])
compare = pd.DataFrame(
    {
        "ols_per_sd": coef["estimate"].drop("const") * scale,
        "elastic_net_per_sd": pen["estimate"].drop("const", errors="ignore"),
    }
)
compare.sort_values("ols_per_sd", key=abs, ascending=False).round(3)

,ols_per_sd,elastic_net_per_sd
term,,
speed_through_water_kn,42.464,24.802
wave_height_m,10.075,7.999
wind_speed_kn,-8.334,-0.498
true_wind_speed_kn,8.086,0.609
is_night,7.816,7.711
sea_temp_c,-3.520,-2.943
draft_aft_m,2.780,2.359
draft_fore_m,2.164,2.086
wind_dir_deg,-0.860,-0.791


## Prediction: compare, tune, hold out

The pipeline cross-validates linear, ridge, lasso, elastic net, random forest and gradient
boosting pipelines (each with imputation and scaling inside the pipeline), takes the best by
RMSE, runs a random search over its hyperparameters, and scores the tuned model on the last 20
percent of rows. The split is by position in the voyage rather than random; adjacent one-minute
records are near copies of each other, and a random split would score the model on its own
training neighbours.

In [7]:
best = result.best_model
print("best by CV:", best)
result.leaderboard[["model", "rmse_cv", "mae_cv", "r2_cv", "fit_time_s"]].round(3)

best by CV: gradient_boosting


,model,rmse_cv,mae_cv,r2_cv,fit_time_s
0,gradient_boosting,12.796,10.203,0.937,0.977
1,random_forest,13.086,10.393,0.934,0.539
2,linear,13.584,10.834,0.929,0.022
3,lasso,13.584,10.836,0.929,0.021
4,ridge,13.584,10.834,0.929,0.010
5,elastic_net,13.644,10.877,0.928,0.015


In [8]:
print("holdout metrics")
print(result.holdout_metrics.round(3))
result.tuning_results.sort_values("rank_test_score")[["params", "mean_test_score", "std_test_score"]].head(5)

holdout metrics
rmse      15.802
mae       12.738
r2         0.754
mape       0.050
bias      -8.130
n       1200.000
dtype: float64


,params,mean_test_score,std_test_score
4,"{'model__learning_rate': 0.06903521707993515, ...",-12.673093,0.276798
1,"{'model__learning_rate': 0.06380556092044458, ...",-12.709055,0.323020
10,"{'model__learning_rate': 0.024591256383397344,...",-12.716976,0.298065
0,"{'model__learning_rate': 0.0646642271741456, '...",-12.725606,0.313151
5,"{'model__learning_rate': 0.013449326615130807,...",-12.729755,0.310299


## Which signals the model uses

Permutation importance shuffles one feature at a time on the holdout and records how much the
score drops. Unlike tree impurity importances it is comparable across model types, and it is
measured on data the model did not train on.

In [9]:
top = result.importances.head(10)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top["feature"][::-1], top["importance_mean"][::-1], xerr=top["importance_std"][::-1], color="#4c72b0")
ax.set_xlabel("drop in R^2 when shuffled")
ax.set_title(f"Permutation importance, {best}")
plt.show()

## Residuals along the track

Holdout residuals are attached to the coordinates of each record. A residual that drifts with
position over the last leg points at a missing covariate (current, fouling, a sensor offset)
rather than noise.

In [10]:
res = result.residuals
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(res["observed"].to_numpy(), label="observed", color="#333333")
ax.plot(res["predicted"].to_numpy(), label="predicted", color="#d62728", alpha=0.8)
ax.set_title("Holdout: observed and predicted fuel demand")
ax.set_ylabel("kg/h")
ax.legend()
plt.show()

## Kepler.gl map

The cell below renders the map inline. If the widget shows as blank text, run
`jupyter nbextension enable --py --sys-prefix keplergl` once in the environment and reload the
page; `fortress_gis.viz.kepler.enable_nbextension()` prints the same command. The HTML export in
the last section does not need the extension and opens in any browser.

In [11]:
track = mar.voyage_track(telemetry)
builder = KeplerMapBuilder(title="Voyage residuals", height=550)
builder.add_layer(track, "track", opacity=0.5)
builder.add_layer(
    res,
    "holdout residuals",
    color_field="residual",
    colors=("#2166ac", "#f7f7f7", "#b2182b"),
    color_scale="quantile",
    radius=5,
)
builder.widget() if kepler_available() else print("keplergl not installed")

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(config={'version': 'v1', 'config': {'visState': {'layers': [{'id': 'track', 'type': 'geojson', 'confi…

## Export

The bundle has the voyage track, the telemetry points with the target, and the holdout residual
points with a diverging QML style. Tables (leaderboard, coefficients, importances) go alongside
as CSV.

In [12]:
paths_out = mar.export_artifacts(result, telemetry, EXPORTS, name="maritime")
for k, v in paths_out.items():
    print(f"{k:>12}: {v.relative_to(ROOT)}")

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


Map saved to /mnt/c/Users/joogl/OneDrive/Documents/VisualStudioCodeProjects/Fortress-GIS-Projects/exports/maritime/maritime_kepler.html!
        qgis: exports/maritime/qgis
 kepler_html: exports/maritime/maritime_kepler.html
kepler_config: exports/maritime/maritime_kepler_config.json
